# 09 · Pseudotime of oligodendrocyte maturation (PAGA + diffusion pseudotime) — GSE216146

Adult brain cell types are mostly terminal, so pseudotime is computed only on a lineage with intermediate states.
The planned OPC -> oligodendrocyte trajectory is **not supported by the data**: in the kNN graph OPC are disconnected
from all oligodendrocytes (PAGA connectivity ~0; *Enpp6* in 1 % of OPC, i.e. no committed OPC captured). Section 1
shows this. The trajectory analysed is therefore **newly formed -> mature (myelinating) oligodendrocyte**
(`pseudotime.lineage`), which is connected; chemotherapy-related cognitive impairment has been linked to
oligodendrocyte / myelin damage, so the question is whether cells are distributed differently along this maturation axis.
The two mature clusters (Oligodendrocyte 1 / 2) are probably two mature states rather than a strict sequence.

Steps: PAGA check with OPC, diffusion map on the global Harmony PCs, **DPT** rooted in the newly formed cell with the
highest *Enpp6*, marker gradients, per-sample comparison (median pseudotime and stage fractions, OLS
`~ batch + condition`), genes changing along pseudotime. The lineage cells are exported with the authors'
`spliced` / `unspliced` layers for RNA velocity in `10_velocity` (separate env `velocity`, see CLAUDE.md).

**Input:** `results/03_annotated.h5ad`, `$DATA/GSE216146_chemo_brain.h5ad.gz` (spliced/unspliced only) ·
**Output:** `results/09_pseudotime_obs.tsv.gz`, `results/09_lineage_velocity_input.h5ad`, tables `results/tables/09_*`,
plots `results/plots/09_*`.

In [ ]:
import gc
import sys
from pathlib import Path

import anndata as ad
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scanpy as sc
import seaborn as sns

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "scripts"))
import sc_utils as scu

cfg = scu.load_config(ROOT / "config.yaml")
p = cfg["pseudotime"]
seed = cfg["integration"]["random_state"]
scu.setup_plotting()
pd.set_option("display.max_rows", 200)
p

## 1. Why OPC are left out: PAGA on OPC + oligodendrocytes

In [ ]:
adata = sc.read_h5ad(cfg["paths"]["results_dir"] / "03_annotated.h5ad")
ctx = adata[adata.obs["cell_type_fine"].isin(p["context"])].copy()
ctx.obs["stage"] = pd.Categorical(ctx.obs["cell_type_fine"].astype(str), categories=p["context"])
sc.pp.neighbors(ctx, n_neighbors=p["n_neighbors"], use_rep=p["use_rep"], random_state=seed)
sc.tl.paga(ctx, groups="stage")
paga_conn = pd.DataFrame(ctx.uns["paga"]["connectivities"].toarray(), index=p["context"], columns=p["context"])
paga_conn.round(3).to_csv(cfg["paths"]["tables_dir"] / "09_paga_connectivity.tsv", sep="\t")
sc.pl.paga(ctx, threshold=0.01, show=False, frameon=False)
scu.save_fig(plt.gcf(), cfg, "09_paga")
plt.show()
del ctx
display(paga_conn.round(3))

lin = adata[adata.obs["cell_type_fine"].isin(p["lineage"])].copy()
del adata
gc.collect()
lin.obs["stage"] = pd.Categorical(lin.obs["cell_type_fine"].astype(str), categories=p["lineage"])
for k in [k for k in lin.uns if k.endswith("_colors") or k in ("neighbors", "umap")]:
    del lin.uns[k]
lin.obsp = {}
print(lin.obs["stage"].value_counts().to_dict())

## 2. Diffusion pseudotime on newly formed -> mature oligodendrocytes

In [ ]:
sc.pp.neighbors(lin, n_neighbors=p["n_neighbors"], use_rep=p["use_rep"], random_state=seed)
sc.tl.umap(lin, random_state=seed)
sc.tl.diffmap(lin, n_comps=p["n_dcs"] + 1, random_state=seed)

# Root: cell of root_type with the highest summed z-score of the root markers
z = lambda g: (lambda x: (x - x.mean()) / x.std())(lin[:, g].X.toarray().ravel())
root_score = sum(z(g) for g in p["root_markers"])
is_root_type = (lin.obs["stage"] == p["root_type"]).to_numpy()
lin.uns["iroot"] = int(np.flatnonzero(is_root_type)[np.argmax(root_score[is_root_type])])
sc.tl.dpt(lin, n_dcs=p["n_dcs"])
order = lin.obs.groupby("stage", observed=True)["dpt_pseudotime"].median()
print("median pseudotime per stage:", order.round(3).to_dict())
assert np.isfinite(lin.obs["dpt_pseudotime"]).all(), "disconnected graph"
assert order.loc[p["root_type"]] < order.drop(p["root_type"]).min(), "root stage is not the earliest"

In [ ]:
fig = sc.pl.umap(lin, color=["stage", "dpt_pseudotime", "condition", "batch"], ncols=2, wspace=0.4, size=12,
                 return_fig=True)
scu.save_fig(fig, cfg, "09_umap_pseudotime")
plt.show()

In [ ]:
# Marker gradients along pseudotime (sanity check of the direction)
markers = [g for g in p["markers"] if g in lin.var_names]
bins = pd.qcut(lin.obs["dpt_pseudotime"].rank(method="first"), p["n_bins"], labels=False)
expr = pd.DataFrame(lin[:, markers].X.toarray(), columns=markers, index=lin.obs_names)
trend = expr.groupby(bins.values).mean()
trend = (trend - trend.min()) / (trend.max() - trend.min())
fig, ax = plt.subplots(figsize=(7, 0.35 * len(markers) + 1.5))
sns.heatmap(trend.T, cmap="viridis", ax=ax, cbar_kws={"label": "scaled mean expression"})
ax.set_xlabel(f"pseudotime bin (1 = earliest, {p['n_bins']} = latest)"); ax.set_ylabel("")
scu.save_fig(fig, cfg, "09_marker_trends")
plt.show()

## 3. Chemotherapy vs control along the trajectory

Per sample (>= `min_cells_per_sample` lineage cells): median pseudotime and the fraction of cells in each stage;
OLS `~ batch + condition` (n = 3 vs 3, exploratory).

In [ ]:
obs = lin.obs
meta = (obs[["biosample", "condition", "batch"]].drop_duplicates().set_index("biosample")
        .sort_values(["condition", "batch"]))
meta.index = meta.index.astype(str)
meta["condition"] = meta["condition"].cat.remove_unused_categories()
meta["batch"] = meta["batch"].astype(str)
n = obs.groupby("biosample", observed=True).size()
keep = n.index[n >= p["min_cells_per_sample"]].astype(str)

per_sample = pd.concat([
    obs.groupby("biosample", observed=True)["dpt_pseudotime"].median().rename("median_pseudotime"),
    pd.crosstab(obs["biosample"], obs["stage"], normalize="index").add_prefix("frac_"),
], axis=1)
per_sample.index = per_sample.index.astype(str)
per_sample = per_sample.loc[keep]
test = scu.ols_by_column(per_sample, meta, p["formula"])
per_sample.join(meta).round(3).to_csv(cfg["paths"]["tables_dir"] / "09_per_sample.tsv", sep="\t")
test.to_csv(cfg["paths"]["tables_dir"] / "09_condition_test.tsv", sep="\t")
display(per_sample.join(meta).round(3))
test.round(4)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
sns.violinplot(obs, x="biosample", y="dpt_pseudotime", hue="condition", ax=axes[0], cut=0, inner="quartile",
               order=list(meta.index), palette={"Control": "#4c72b0", "Chemotherapy": "#dd8452"})
axes[0].tick_params(axis="x", labelrotation=30); axes[0].set_title("Pseudotime distribution per sample")
for cond, color in [("Control", "#4c72b0"), ("Chemotherapy", "#dd8452")]:
    for s in meta.index[meta["condition"] == cond]:
        x = np.sort(obs.loc[obs["biosample"] == s, "dpt_pseudotime"])
        axes[1].plot(x, np.arange(1, len(x) + 1) / len(x), c=color, lw=1.2, alpha=0.8,
                     label=cond if s == meta.index[meta["condition"] == cond][0] else None)
axes[1].set_xlabel("diffusion pseudotime"); axes[1].set_ylabel("cumulative fraction of cells")
axes[1].set_title("ECDF per sample"); axes[1].legend(frameon=False)
for ax in axes:
    ax.grid(False)
fig.tight_layout()
scu.save_fig(fig, cfg, "09_pseudotime_by_sample")
plt.show()

## 4. Genes changing along pseudotime

Spearman correlation of each gene with pseudotime in **control** cells (genes detected in >= 10 % of lineage cells),
then the trend of the top genes in each condition (descriptive).

In [ ]:
from scipy.stats import spearmanr

det = np.asarray((lin.X > 0).mean(axis=0)).ravel() >= 0.1
ctrl = (obs["condition"] == "Control").to_numpy()
X = lin[ctrl][:, det].X.toarray()
pt = obs.loc[ctrl, "dpt_pseudotime"].to_numpy()
rho = spearmanr(pt, X).statistic[0, 1:]
dyn = pd.Series(rho, index=lin.var_names[det]).dropna().sort_values()
top = list(dyn.index[: p["n_dynamic_genes"] // 2]) + list(dyn.index[-(p["n_dynamic_genes"] // 2):])
dyn.rename("spearman_rho").to_frame().to_csv(cfg["paths"]["tables_dir"] / "09_dynamic_genes.tsv", sep="\t")
print("decreasing:", list(dyn.index[:10])); print("increasing:", list(dyn.index[-10:][::-1]))

expr = pd.DataFrame(lin[:, top].X.toarray(), columns=top, index=lin.obs_names)
fig, axes = plt.subplots(1, 2, figsize=(12, 0.22 * len(top) + 2), sharey=True)
for ax, cond in zip(axes, ["Control", "Chemotherapy"]):
    m = (obs["condition"] == cond).to_numpy()
    tr = expr[m].groupby(bins.values[m]).mean()
    tr = (tr - expr.groupby(bins.values).mean().min()) / (expr.groupby(bins.values).mean().max() - expr.groupby(bins.values).mean().min())
    sns.heatmap(tr.T, cmap="viridis", vmin=0, vmax=1, ax=ax, cbar=ax is axes[1], cbar_kws={"label": "scaled mean"})
    ax.set_title(cond); ax.set_xlabel("pseudotime bin")
axes[0].tick_params(axis="y", labelsize=7)
fig.tight_layout()
scu.save_fig(fig, cfg, "09_dynamic_genes_by_condition")
plt.show()

## 5. Export for RNA velocity (`10_velocity`)

Only the `spliced` / `unspliced` layers of the lineage cells (and our filtered genes) are read from the authors' h5ad
(decompressed once into `results/`) with `scu.read_h5ad_layers`; cell barcodes match the counts files exactly (01).

In [ ]:
genes = lin.var_names
layers = scu.read_h5ad_layers(scu.decompress_author_h5ad(cfg), ["spliced", "unspliced"], lin.obs_names, genes)
vel = ad.AnnData(X=layers["spliced"].copy(), layers=layers,
                 obs=lin.obs[["biosample", "condition", "batch", "stage", "dpt_pseudotime"]].copy(),
                 var=pd.DataFrame(index=genes))
for key in ("X_umap", p["use_rep"]):
    vel.obsm[key] = lin.obsm[key]
vel.write_h5ad(cfg["paths"]["results_dir"] / p["velocity_input"], compression="gzip")
lin.obs[["biosample", "condition", "batch", "stage", "dpt_pseudotime"]].to_csv(
    cfg["paths"]["results_dir"] / "09_pseudotime_obs.tsv.gz", sep="\t")
vel

In [ ]:
del lin, vel, layers
_ = gc.collect()